In [0]:
# Use workspace catalog 
CATALOG = "workspace"
SCHEMA = "default"

# Fully-qualified table references
TBL_DAY1     = f"{CATALOG}.{SCHEMA}.day1_tickets"
TBL_DAY2     = f"{CATALOG}.{SCHEMA}.day2_tickets"
TBL_PROFILES = f"{CATALOG}.{SCHEMA}.agent_profiles"

print(f"✅ Target Paths Initialized:\n  {TBL_DAY1}\n  {TBL_DAY2}\n  {TBL_PROFILES}")

✅ Target Paths Initialized:
  workspace.default.day1_tickets
  workspace.default.day2_tickets
  workspace.default.agent_profiles


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# 1. Generate Agent Profiles Data (TL01–TL08 + Out of scope)
profiles_data = [
    ("A001","Agent_A001","Junior Support Agent","TL01"), ("A002","Agent_A002","Senior Support Agent","TL01"),
    ("A003","Agent_A003","Senior Support Agent","TL01"), ("A004","Agent_A004","Junior Support Agent","TL01"),
    ("A005","Agent_A005","Support Agent","TL01"),       ("A006","Agent_A006","Support Agent","TL02"),
    ("A007","Agent_A007","Junior Support Agent","TL02"), ("A008","Agent_A008","Senior Support Agent","TL02"),
    ("A009","Agent_A009","Support Agent","TL02"),       ("A010","Agent_A010","Junior Support Agent","TL02"),
    ("A011","Agent_A011","Support Agent","TL03"),       ("A012","Agent_A012","Senior Support Agent","TL03"),
    ("A013","Agent_A013","Junior Support Agent","TL03"), ("A014","Agent_A014","Support Agent","TL03"),
    ("A015","Agent_A015","Senior Support Agent","TL03"), ("A016","Agent_A016","Junior Support Agent","TL04"),
    ("A017","Agent_A017","Support Agent","TL04"),       ("A018","Agent_A018","Senior Support Agent","TL04"),
    ("A019","Agent_A019","Support Agent","TL04"),       ("A020","Agent_A020","Junior Support Agent","TL04"),
    ("A021","Agent_A021","Senior Support Agent","TL05"), ("A022","Agent_A022","Support Agent","TL05"),
    ("A023","Agent_A023","Junior Support Agent","TL05"), ("A024","Agent_A024","Support Agent","TL05"),
    ("A025","Agent_A025","Senior Support Agent","TL05"), ("A026","Agent_A026","Support Agent","TL06"),
    ("A027","Agent_A027","Junior Support Agent","TL06"), ("A028","Agent_A028","Senior Support Agent","TL06"),
    ("A029","Agent_A029","Support Agent","TL06"),       ("A030","Agent_A030","Junior Support Agent","TL06"),
    ("A031","Agent_A031","Senior Support Agent","TL07"), ("A032","Agent_A032","Support Agent","TL07"),
    ("A033","Agent_A033","Junior Support Agent","TL07"), ("A034","Agent_A034","Support Agent","TL07"),
    ("A035","Agent_A035","Senior Support Agent","TL07"), ("A036","Agent_A036","Junior Support Agent","TL08"),
    ("A037","Agent_A037","Support Agent","TL08"),       ("A038","Agent_A038","Senior Support Agent","TL08"),
    ("A039","Agent_A039","Support Agent","TL08"),       ("A040","Agent_A040","Junior Support Agent","TL08"),
    ("A041","Agent_A041","Support Agent","TL09"),       ("A042","Agent_A042","Support Agent","TL12")
]
profiles_schema = StructType([
    StructField("agent_id", StringType(), True), StructField("agent_name", StringType(), True),
    StructField("role", StringType(), True),       StructField("team_lead_id", StringType(), True)
])
spark.createDataFrame(profiles_data, schema=profiles_schema).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TBL_PROFILES)

# 2. Generate Day 1 Ticket Logs
day1_data = [
    ("TKT00001","A001","Resolved","0h 35m 45s","Technical"), ("TKT00002","A001","Resolved","0h 22m 10s","Billing"),
    ("TKT00003","A001","Resolved","0h 12m 5s", "General"),   ("TKT00004","A001","Open",    "0h 40m 0s", "Delivery"),
    ("TKT00005","A002","Resolved","1h 10m 30s","Returns"),    ("TKT00006","A002","Resolved","0h 16m 26s","Billing"),
    ("TKT00007","A002","Pending", "0h 36m 0s", "Returns"),    ("TKT00008","A003","Resolved","0h 25m 50s","Technical"),
    ("TKT00009","A003","Resolved","0h 15m 0s", "Billing"),    ("TKT00010","A003","Resolved","0h 18m 44s","Account"),
    ("TKT00011","A004","Resolved","0h 33m 15s","Delivery"),   ("TKT00012","A004","Open",    "0h 10m 0s", "General"),
    ("TKT00013","A004","Resolved","0h 20m 30s","Returns"),    ("TKT00014","A005","Resolved","0h 45m 0s", "Account"),
    ("TKT00015","A005","Resolved","0h 14m 59s","Technical"),  ("TKT00016","A005","Pending", "0h 55m 0s", "Billing"),
    ("TKT00017","A006","Resolved","0h 28m 20s","Technical"),  ("TKT00018","A006","Resolved","0h 17m 45s","Delivery"),
    ("", "A001","Resolved","0h 20m 0s", "Billing"),          ("TKT99990", "","Resolved","0h 25m 0s", "Technical")
]
ticket_schema = StructType([
    StructField("ticket_id", StringType(), True),       StructField("agent_id", StringType(), True),
    StructField("status", StringType(), True),          StructField("resolution_time", StringType(), True),
    StructField("category", StringType(), True)
])
spark.createDataFrame(day1_data, schema=ticket_schema).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TBL_DAY1)

# 3. Generate Day 2 Ticket Logs
day2_data = [
    ("TKT00200","A001","Resolved","0h 28m 0s", "Billing"),   ("TKT00201","A001","Resolved","0h 16m 30s","Technical"),
    ("TKT00202","A002","Resolved","0h 35m 0s", "Account"),   ("TKT00203","A002","Open",    "0h 22m 0s", "Delivery"),
    ("TKT00204","A003","Resolved","0h 20m 45s","Returns"),   ("TKT00205","A003","Resolved","0h 9m 0s",  "General"),
    ("", "A010","Resolved","0h 30m 0s","Billing"),          ("TKT99993", "A011","Resolved","", "Returns")
]
spark.createDataFrame(day2_data, schema=ticket_schema).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TBL_DAY2)

print("✅ Injected Simulation Data Directly Into DBFS Database Layer.")

✅ Injected Simulation Data Directly Into DBFS Database Layer.


In [0]:
# Load raw configurations from Bronze layer
bronze_day1 = spark.table(TBL_DAY1).withColumn("Day", F.lit(1).cast("integer"))
bronze_day2 = spark.table(TBL_DAY2).withColumn("Day", F.lit(2).cast("integer"))
bronze_profiles = spark.table(TBL_PROFILES)

def drop_critical_nulls(df):
    return df.filter(
        F.col("ticket_id").isNotNull() & (F.col("ticket_id") != "") &
        F.col("agent_id").isNotNull() & (F.col("agent_id") != "") &
        F.col("resolution_time").isNotNull() & (F.col("resolution_time") != "")
    )

clean_day1 = drop_critical_nulls(bronze_day1)
clean_day2 = drop_critical_nulls(bronze_day2)
print(f"🧹 Data Quality Pass Finished.")

🧹 Data Quality Pass Finished.


In [0]:
import re

def parse_resolution_time(time_str):
    if not time_str: return None
    m = re.match(r"(\d+)\s*h\s*(\d+)\s*m\s*(\d+)\s*s", time_str.strip(), re.IGNORECASE)
    if not m: return None
    h, mins, secs = int(m.group(1)), int(m.group(2)), int(m.group(3))
    total = h * 60 + mins
    if secs >= 30: total += 1
    return total

parse_time_udf = F.udf(parse_resolution_time, IntegerType())

silver_day1_times = clean_day1.withColumn("status_clean", F.upper(F.trim(F.col("status")))).withColumn("resolved_minutes", parse_time_udf(F.col("resolution_time"))).filter(F.col("resolved_minutes").isNotNull())
silver_day2_times = clean_day2.withColumn("status_clean", F.upper(F.trim(F.col("status")))).withColumn("resolved_minutes", parse_time_udf(F.col("resolution_time"))).filter(F.col("resolved_minutes").isNotNull())

In [0]:
IN_SCOPE_TLS = {f"TL{str(i).zfill(2)}" for i in range(1, 9)}

def process_filters(tickets_df, profiles_df):
    enriched = tickets_df.join(F.broadcast(profiles_df), on="agent_id", how="inner")
    scoped = enriched.filter(F.col("team_lead_id").isin(IN_SCOPE_TLS))
    # Keep strictly higher than 15 minutes and matching 'RESOLVED' status
    return scoped.filter((F.col("status_clean") == "RESOLVED") & (F.col("resolved_minutes") > 15))

silver_day1_success = process_filters(silver_day1_times, bronze_profiles)
silver_day2_success = process_filters(silver_day2_times, bronze_profiles)

In [0]:
# Identify successful agents from Day 1 to drop from Day 2 data execution
day1_successful_agents = silver_day1_success.select("agent_id").distinct()
silver_day2_carryover = silver_day2_success.join(day1_successful_agents, on="agent_id", how="left_anti")

# Combine results
gold_combined = silver_day1_success.union(silver_day2_carryover)

print("=== Q1 — Resolution Rates by Team Lead ===")
gold_q1 = gold_combined.groupBy("team_lead_id").agg(F.count("ticket_id").alias("total_resolved_tickets")).orderBy(F.col("total_resolved_tickets").desc())
display(gold_q1)

print("\n=== Q2 — Per-Agent Summary Matrix ===")
gold_q2 = gold_combined.groupBy("agent_id", "agent_name", "Day").agg(F.count("ticket_id").alias("resolved_count")).orderBy("agent_id")
display(gold_q2)

=== Q1 — Resolution Rates by Team Lead ===


team_lead_id,total_resolved_tickets
TL01,9
TL02,2



=== Q2 — Per-Agent Summary Matrix ===


agent_id,agent_name,Day,resolved_count
A001,Agent_A001,1,2
A002,Agent_A002,1,2
A003,Agent_A003,1,2
A004,Agent_A004,1,2
A005,Agent_A005,1,1
A006,Agent_A006,1,2


In [0]:
# Validate Team Lead distribution
display(bronze_profiles.groupBy("team_lead_id").count().orderBy("team_lead_id"))

# Validate Day 1 ticket statuses
display(bronze_day1.groupBy("status").count())

# Validate Day 2 ticket statuses
display(bronze_day2.groupBy("status").count())

# Final Gold Report 1
display(gold_q1)

# Final Gold Report 2
display(gold_q2.orderBy("Day", "agent_id"))

# Record counts
print("Profiles:", bronze_profiles.count())
print("Day1:", bronze_day1.count())
print("Day2:", bronze_day2.count())
print("Gold TL:", gold_q1.count())
print("Gold Agent:", gold_q2.count())

team_lead_id,count
TL01,5
TL02,5
TL03,5
TL04,5
TL05,5
TL06,5
TL07,5
TL08,5
TL09,1
TL12,1


status,count
Resolved,16
Open,2
Pending,2


status,count
Resolved,7
Open,1


team_lead_id,total_resolved_tickets
TL01,9
TL02,2


agent_id,agent_name,Day,resolved_count
A001,Agent_A001,1,2
A002,Agent_A002,1,2
A003,Agent_A003,1,2
A004,Agent_A004,1,2
A005,Agent_A005,1,1
A006,Agent_A006,1,2


Profiles: 42
Day1: 20
Day2: 8
Gold TL: 2
Gold Agent: 6
